In [ ]:
import pandas as pd
import numpy as np

# input
gen_database_path = snakemake.input.gen_database_path
store_database_path = snakemake.input.store_database_path

# output
gen_comm_decomm_path = snakemake.output.gen_comm_decomm_path
store_comm_decomm_path = snakemake.output.store_comm_decomm_path

# parameters
model_years = snakemake.params.model_years

inputs = [gen_database_path,store_database_path]
outputs = [gen_comm_decomm_path, store_comm_decomm_path]


In [ ]:
for input_file, output_file in zip(inputs,outputs):

    comm_decomm = pd.read_csv(
        input_file
    )

    commissioning_plan = (
        comm_decomm
        .query("commissioning_year <= @model_years[1]")
        .query("decommissioning_year > @model_years[0]")
        .drop(columns=["decommissioning_year"])
        .replace({np.nan:""})
        .groupby(["Technology","zone","region","commissioning_year"]).sum()
        .reset_index()
        .rename(columns={"commissioning_year":"Year"})
        .assign(type="commissioning_year")
    )

    decommissioning_plan = (
        comm_decomm
        .query("decommissioning_year > @model_years[0]")
        .query("commissioning_year <= @model_years[1]")
        .drop(columns=["commissioning_year"])
        .replace({np.nan:""})
        .groupby(["Technology","zone","region","decommissioning_year"]).sum()
        .reset_index()
        .rename(columns={"decommissioning_year":"Year"})
        .assign(type="decommissioning_year")           
    )

    pd.concat([commissioning_plan,decommissioning_plan]).to_csv(
        output_file,
        index=False,
    )